In [1]:
import logging
import platform
import random
import subprocess
import time
from os import getenv

import fs_structs
from dotenv import load_dotenv

from distributed_processing.async_result import gather
from distributed_processing.utils import fsclient

In [2]:
load_dotenv()
NS_PATH = getenv("NS_PATH")
MASTER_QUEUE = getenv("MASTER_QUEUE")
LAUNCH_SERVER = False

MASTER_QUEUE

'node_1'

In [3]:
# logging.getLogger("distributed_processing").setLevel(logging.DEBUG)
# logging.getLogger("distributed_processing.filesystem_connector").setLevel(logging.DEBUG)

In [4]:
if LAUNCH_SERVER:
    PYTHON_INTERPRETER = getenv("PYTHON_INTERPRETER")
    MULTI_WORKER_SCRIPT = getenv("MULTI_WORKER_SCRIPT")
    server_process = subprocess.Popen(
        [PYTHON_INTERPRETER, MULTI_WORKER_SCRIPT],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        shell=False,
    )
    time.sleep(10)

In [3]:
# NBVAL_CHECK_OUTPUT
"""
fs_connector = FileSystemConnector(NS_PATH)
fs_connector.with_watchdog=True
fs_connector.pop_watchdog_timeout = 10

client = Client(DummySerializer(), fs_connector, check_registry="cache")
"""

client = fsclient(NS_PATH)
client.timeout = 30

Client with id: fs_client_4
Results queue: fs_client_4_responses


In [4]:
MASTER_QUEUE = client.to_requests_queue(MASTER_QUEUE)

In [5]:
client.update_registry_cache()

In [6]:
ns = fs_structs.structs.FSNamespace(NS_PATH)

In [7]:
# NBVAL_CHECK_OUTPUT
ns.registry.keys()

['method_queues_add',
 'method_queues_cleanup',
 'method_queues_create_worker',
 'method_queues_dic',
 'method_queues_div',
 'method_queues_eval_py_function',
 'method_queues_hola',
 'method_queues_info',
 'method_queues_kill_all_processes',
 'method_queues_kill_process',
 'method_queues_kill_processes',
 'method_queues_list_processes',
 'method_queues_lista',
 'method_queues_mul',
 'method_queues_sleep',
 'method_queues_tupla',
 'nclients',
 'nservers',
 'workers_queue_requests_cola_1',
 'workers_queue_requests_cola_2',
 'workers_queue_requests_fs_server_4',
 'workers_queue_requests_fs_server_5',
 'workers_queue_requests_fs_server_6',
 'workers_queue_requests_node_1',
 'workers_queue_requests_py_eval']

In [10]:
z1 = client.rpc_sync("create_worker", ["worker1"], queue=MASTER_QUEUE)
z2 = client.rpc_sync("create_worker", ["worker1"], queue=MASTER_QUEUE)
z3 = client.rpc_sync("create_worker", ["worker1"], queue=MASTER_QUEUE)

(z1, z2, z3)

(42896, 9488, 42372)

In [11]:
ps = client.rpc_sync("list_processes", [], queue=MASTER_QUEUE)
ps

[(42896, 'worker1', 'fs_server_1'),
 (9488, 'worker1', 'fs_server_2'),
 (42372, 'worker1', 'fs_server_3')]

In [12]:
# NBVAL_CHECK_OUTPUT

len(ps)

3

In [13]:
# NBVAL_CHECK_OUTPUT

[(w, q) for _, w, q in ps]

[('worker1', 'fs_server_1'),
 ('worker1', 'fs_server_2'),
 ('worker1', 'fs_server_3')]

In [14]:
# NBVAL_CHECK_OUTPUT

client.rpc_sync("kill_process", [z2], queue=MASTER_QUEUE)

(True, False)

In [15]:
client.rpc_sync("list_processes", [], queue=MASTER_QUEUE)

[(42896, 'worker1', 'fs_server_1'), (42372, 'worker1', 'fs_server_3')]

In [16]:
# NBVAL_CHECK_OUTPUT

wps = [pid for pid, w, q in client.rpc_sync("list_processes", [], queue=MASTER_QUEUE)]
len(wps)

2

In [17]:
client.rpc_sync("kill_all_processes", [], queue=MASTER_QUEUE)

[(True, False), (False, False), (True, False)]

In [18]:
# NBVAL_CHECK_OUTPUT

wps = [pid for pid, w, q in client.rpc_sync("list_processes", [], queue=MASTER_QUEUE)]
len(wps)

0

In [19]:
z1 = client.rpc_sync("create_worker", ["worker1"], queue=MASTER_QUEUE)
z2 = client.rpc_sync("create_worker", ["worker1"], queue=MASTER_QUEUE)
z3 = client.rpc_sync("create_worker", ["worker1"], queue=MASTER_QUEUE)

In [20]:
# NBVAL_CHECK_OUTPUT

wps = [pid for pid, w, q in client.rpc_sync("list_processes", [], queue=MASTER_QUEUE)]
len(wps)

3

In [8]:
client.update_registry_cache()

In [9]:
y = client.rpc_async("add", [1, 0])

In [10]:
# NBVAL_CHECK_OUTPUT

y.get()

1

In [11]:
# NBVAL_CHECK_OUTPUT

client.check_registry = "always"

try:
    y = client.rpc_async("fake", [1, 0])
except Exception as e:
    print(e)

'method_queues_fake'


In [12]:
# NBVAL_CHECK_OUTPUT

client.check_registry = "never"  # use queue client.default_queue, by default "default"
client.set_default_queue("cola_1")

y = client.rpc_async("fake", [1, 0])

try:
    y.get()
except Exception as e:
    print(e)


Error -32601 : The method does not exist/is not available.

 NoneType: None



In [13]:
# NBVAL_CHECK_OUTPUT

client.check_registry = "never"
client.set_default_queue("cola_1")

y = client.rpc_async("fake", [1, 0])

y.safe_get(default="Esto es un error")

'Esto es un error'

In [14]:
client.check_registry = "cache"

In [15]:
x = client.rpc_async("div", [1, 0])

In [16]:
try:
    x.get()
except Exception as e:
    print(e)

Error -3260 : Undefined error

 Traceback (most recent call last):
  File "\\ntcimdavinfo2\fondos\arquitectura_gestora\desarrollo\py_distributed_processing\distributed_processing\worker.py", line 362, in process_single_request
    result = func[request["method"]](*args, **kwargs)
  File "G:\arquitectura_gestora\desarrollo\py_distributed_processing\tests\fs_worker_multi.py", line 28, in div
    return x / y
           ~~^~~
ZeroDivisionError: division by zero



In [17]:
# NBVAL_CHECK_OUTPUT

try:
    x.get()
except Exception as e:
    print("Error")

Error


In [18]:
# x = client.rpc_sync("div", [1, 0])

In [19]:
# NBVAL_CHECK_OUTPUT


client.rpc_sync("add", [1, 1])

2

In [20]:
def f(x, y):
    return x + y


y = client.rpc_async_fn(f, [1, 2.0])

In [21]:
y.get()

3.0

In [22]:
# NBVAL_CHECK_OUTPUT

client.rpc_sync_fn(f, [3.0, 2.0])

5.0

In [23]:
tp = []
N = 30
for i in range(N):
    fn = random.choice(("add", "mul", "div", "lista", "tupla", "dic"))
    t = (fn, [random.random(), random.random()], {})
    tp.append(t)

t = ("sleep", [10], {})
tp.append(t)
tp


[('dic', [0.035602084067515305, 0.7211604495984787], {}),
 ('lista', [0.05533082690298474, 0.10842244136871004], {}),
 ('div', [0.48141767625371823, 0.6365181731595982], {}),
 ('dic', [0.05781207567603741, 0.5018394519625198], {}),
 ('add', [0.31848704861101573, 0.48935060669962316], {}),
 ('add', [0.8776249879552275, 0.5108254476138916], {}),
 ('add', [0.37316476817715394, 0.37912981884769714], {}),
 ('mul', [0.2417789742106402, 0.2605137432894923], {}),
 ('div', [0.6288174789655763, 0.0007991699609627423], {}),
 ('dic', [0.28837401764172477, 0.9165111349742212], {}),
 ('dic', [0.8105795201601335, 0.909316458039367], {}),
 ('lista', [0.04251684188696048, 0.03421097351656299], {}),
 ('dic', [0.33919707619529116, 0.9086051644196429], {}),
 ('lista', [0.1644393495355868, 0.943560883164257], {}),
 ('tupla', [0.5257710984070757, 0.8757292768329408], {}),
 ('dic', [0.19777970843455173, 0.7868006676591403], {}),
 ('mul', [0.9121130065404195, 0.5004043196416099], {}),
 ('tupla', [0.7845219175

In [ ]:
tp = [
    ("div", [0.5273328219558507, 0.13835718442890943], {}, "cola_1"),
    ("div", [0.7224042937278776, 0.6744424742714074], {}), "cola_1",
    ("mul", [0.16464192867054384, 0.2961055537758295], {}),
    ("lista", [0.24324779336838243, 0.4486736957376778], {}),
    ("dic", [0.222443603731179, 0.049201002693669005], {}),
    ("dic", [0.5892562777042863, 0.8190093828367946], {}),
    ("div", [0.9698052964451762, 0.2495544466990297], {}),
    ("add", [0.18281717945238662, 0.28456253134154685], {}),
    ("div", [0.8231058337900704, 0.4550312056693214], {}),
    ("mul", [0.6955981505190975, 0.2960636895338262], {}),
    ("add", [0.5793774647414703, 0.9353212122782703], {}),
    ("lista", [0.3893530442489298, 0.74231052966268], {}),
    ("div", [0.6419882052325951, 0.7661892480993476], {}),
    ("div", [0.049994104986677, 0.4562378471461709], {}),
    ("dic", [0.4734342157728231, 0.053714834674179035], {}),
    ("mul", [0.8092977961625194, 0.9195146847049076], {}),
    ("mul", [0.913778636227633, 0.6145608354175943], {}),
    ("lista", [0.7499808955353686, 0.5337360859450593], {}),
    ("dic", [0.6756209555432302, 0.9505296005797351], {}),
    ("dic", [0.5209295316393681, 0.9420323858687962], {}),
    ("div", [0.15809810362813237, 0.62619392590883], {}),
    ("dic", [0.29474022335742067, 0.893515494048087], {}),
    ("mul", [0.5349022233942071, 0.05757455715844495], {}),
    ("lista", [0.042102069906052586, 0.3469740971990074], {}),
    ("mul", [0.871021663889528, 0.007201521377221853], {}),
    ("lista", [0.6049724123450363, 0.26801461728942155], {}),
    ("dic", [0.8898534023208522, 0.5019296231298458], {}),
    ("lista", [0.9266421130454301, 0.9972178178045238], {}),
    ("mul", [0.48602513421859184, 0.199817263488683], {}),
    ("dic", [0.7163791721275343, 0.6950721561324937], {}),
    ("sleep", [10], {}),
]

In [25]:
fs = []
for t in tp:
    fs.append(client.rpc_async(t[0], t[1], t[2]))

In [26]:
# NBVAL_CHECK_OUTPUT

resultados = [f.get() for f in fs]
resultados

[3.8113873459661534,
 1.0711132843587332,
 0.048751389463712,
 [0.24324779336838243, 0.4486736957376778],
 {'a': 0.222443603731179, 'b': [0.222443603731179, 0.049201002693669005]},
 {'a': 0.5892562777042863, 'b': [0.5892562777042863, 0.8190093828367946]},
 3.886147128505352,
 0.4673797107939335,
 1.8088997491487095,
 0.2059413548755898,
 1.5146986770197406,
 [0.3893530442489298, 0.74231052966268],
 0.8378976954129118,
 0.10957903930898512,
 {'a': 0.4734342157728231, 'b': [0.4734342157728231, 0.053714834674179035]},
 0.7441612078707556,
 0.5615725620668042,
 [0.7499808955353686, 0.5337360859450593],
 {'a': 0.6756209555432302, 'b': [0.6756209555432302, 0.9505296005797351]},
 {'a': 0.5209295316393681, 'b': [0.5209295316393681, 0.9420323858687962]},
 0.2524746681288481,
 {'a': 0.29474022335742067, 'b': [0.29474022335742067, 0.893515494048087]},
 0.03079675863498907,
 [0.042102069906052586, 0.3469740971990074],
 0.006272681132523783,
 [0.6049724123450363, 0.26801461728942155],
 {'a': 0.8898

In [ ]:
# NBVAL_CHECK_OUTPUT

fs = client.rpc_multi_async(tp)

In [28]:
time.sleep(3)

In [29]:
# NBVAL_CHECK_OUTPUT

# AsynResult.status updates the information every time it is called.
# 'PENDING' state should be assumed as transitory.
# If there are responses available in the queue or in the cache
# status or pending() updates the AsyncResult object.

[f.status for f in fs]

['OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'PENDING']

In [30]:
r = client.wait_responses(timeout=2)
r

['fs_client_4:69']

In [31]:
# NBVAL_CHECK_OUTPUT

len(r)

1

In [32]:
time.sleep(7)

In [33]:
# NBVAL_CHECK_OUTPUT
# AsynResult.status updates information

[f.status for f in fs]

['OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK']

In [34]:
# NBVAL_CHECK_OUTPUT

client.wait_responses()

[]

In [35]:
# NBVAL_CHECK_OUTPUT

try:
    client.wait_responses(["kk", "qq"])
except ValueError as e:
    print(e)

wait_responses: ['kk', 'qq'] neither in responses nor in pending.


In [36]:
client._update_cache_with_all_available_responses()

In [37]:
# NBVAL_CHECK_OUTPUT

client.pending

{}

In [38]:
# NBVAL_CHECK_OUTPUT

[f.get() for f in fs]

[3.8113873459661534,
 1.0711132843587332,
 0.048751389463712,
 [0.24324779336838243, 0.4486736957376778],
 {'a': 0.222443603731179, 'b': [0.222443603731179, 0.049201002693669005]},
 {'a': 0.5892562777042863, 'b': [0.5892562777042863, 0.8190093828367946]},
 3.886147128505352,
 0.4673797107939335,
 1.8088997491487095,
 0.2059413548755898,
 1.5146986770197406,
 [0.3893530442489298, 0.74231052966268],
 0.8378976954129118,
 0.10957903930898512,
 {'a': 0.4734342157728231, 'b': [0.4734342157728231, 0.053714834674179035]},
 0.7441612078707556,
 0.5615725620668042,
 [0.7499808955353686, 0.5337360859450593],
 {'a': 0.6756209555432302, 'b': [0.6756209555432302, 0.9505296005797351]},
 {'a': 0.5209295316393681, 'b': [0.5209295316393681, 0.9420323858687962]},
 0.2524746681288481,
 {'a': 0.29474022335742067, 'b': [0.29474022335742067, 0.893515494048087]},
 0.03079675863498907,
 [0.042102069906052586, 0.3469740971990074],
 0.006272681132523783,
 [0.6049724123450363, 0.26801461728942155],
 {'a': 0.8898

In [39]:
fs = client.rpc_batch_async(tp)

In [40]:
# NBVAL_CHECK_OUTPUT

[f.get() for f in fs]

[3.8113873459661534,
 1.0711132843587332,
 0.048751389463712,
 [0.24324779336838243, 0.4486736957376778],
 {'a': 0.222443603731179, 'b': [0.222443603731179, 0.049201002693669005]},
 {'a': 0.5892562777042863, 'b': [0.5892562777042863, 0.8190093828367946]},
 3.886147128505352,
 0.4673797107939335,
 1.8088997491487095,
 0.2059413548755898,
 1.5146986770197406,
 [0.3893530442489298, 0.74231052966268],
 0.8378976954129118,
 0.10957903930898512,
 {'a': 0.4734342157728231, 'b': [0.4734342157728231, 0.053714834674179035]},
 0.7441612078707556,
 0.5615725620668042,
 [0.7499808955353686, 0.5337360859450593],
 {'a': 0.6756209555432302, 'b': [0.6756209555432302, 0.9505296005797351]},
 {'a': 0.5209295316393681, 'b': [0.5209295316393681, 0.9420323858687962]},
 0.2524746681288481,
 {'a': 0.29474022335742067, 'b': [0.29474022335742067, 0.893515494048087]},
 0.03079675863498907,
 [0.042102069906052586, 0.3469740971990074],
 0.006272681132523783,
 [0.6049724123450363, 0.26801461728942155],
 {'a': 0.8898

In [41]:
# NBVAL_CHECK_OUTPUT

client.rpc_batch_sync(tp)

[3.8113873459661534,
 1.0711132843587332,
 0.048751389463712,
 [0.24324779336838243, 0.4486736957376778],
 {'a': 0.222443603731179, 'b': [0.222443603731179, 0.049201002693669005]},
 {'a': 0.5892562777042863, 'b': [0.5892562777042863, 0.8190093828367946]},
 3.886147128505352,
 0.4673797107939335,
 1.8088997491487095,
 0.2059413548755898,
 1.5146986770197406,
 [0.3893530442489298, 0.74231052966268],
 0.8378976954129118,
 0.10957903930898512,
 {'a': 0.4734342157728231, 'b': [0.4734342157728231, 0.053714834674179035]},
 0.7441612078707556,
 0.5615725620668042,
 [0.7499808955353686, 0.5337360859450593],
 {'a': 0.6756209555432302, 'b': [0.6756209555432302, 0.9505296005797351]},
 {'a': 0.5209295316393681, 'b': [0.5209295316393681, 0.9420323858687962]},
 0.2524746681288481,
 {'a': 0.29474022335742067, 'b': [0.29474022335742067, 0.893515494048087]},
 0.03079675863498907,
 [0.042102069906052586, 0.3469740971990074],
 0.006272681132523783,
 [0.6049724123450363, 0.26801461728942155],
 {'a': 0.8898

In [42]:
tp = []
N = 30
for i in range(N):
    fn = random.choice(("add", "mul", "div", "fake"))
    t = (fn, [random.random(), random.random()], {})
    tp.append(t)

tp

[('mul', [0.5534418134842373, 0.3892674528334199], {}),
 ('fake', [0.2111659460580313, 0.5299259355293667], {}),
 ('fake', [0.15106140242006727, 0.865852211577628], {}),
 ('div', [0.9497658166461254, 0.8584539619461625], {}),
 ('add', [0.09042210061880684, 0.1686247033206847], {}),
 ('mul', [0.9493131067943346, 0.7195621978099871], {}),
 ('div', [0.6862877453072617, 0.6340421258216143], {}),
 ('fake', [0.03155935687396361, 0.7961265744241203], {}),
 ('div', [0.19620267122859603, 0.6633345389251388], {}),
 ('fake', [0.03515300547520439, 0.5704638277248008], {}),
 ('fake', [0.4817774419922676, 0.4784251165615092], {}),
 ('mul', [0.12926678851082174, 0.4856037463678299], {}),
 ('add', [0.3624858164451896, 0.15546512956969338], {}),
 ('div', [0.24214006158446755, 0.8463807421991945], {}),
 ('fake', [0.8227531570794467, 0.2063246643341352], {}),
 ('mul', [0.4143805985870217, 0.39861527727551704], {}),
 ('add', [0.11414116660612195, 0.4523439616668653], {}),
 ('fake', [0.12247060948509458, 0

In [43]:
tp = [
    ("mul", [0.7355407026565478, 0.023519777553893007], {}),
    ("fake", [0.2520522260048764, 0.28054227055242864], {}),
    ("mul", [0.8838106264839363, 0.19639888038506714], {}),
    ("mul", [0.8857412900943293, 0.1253016179972587], {}),
    ("add", [0.0005226378683395039, 0.6568308199617323], {}),
    ("add", [0.15476536347494607, 0.4492869825171584], {}),
    ("fake", [0.7631067253940333, 0.019049359538004906], {}),
    ("fake", [0.4310102131915736, 0.675491507770126], {}),
    ("fake", [0.7140511506543913, 0.7837833237953286], {}),
    ("add", [0.0909525538014786, 0.44959184616881276], {}),
    ("add", [0.6627327150574825, 0.7401973301261011], {}),
    ("div", [0.21232540669537237, 0.7667374101737603], {}),
    ("add", [0.887254961441083, 0.21290364755712576], {}),
    ("fake", [0.7372649371990656, 0.3796617846297834], {}),
    ("add", [0.30864649241428155, 0.8777033159855755], {}),
    ("div", [0.4997997676145608, 0.45884184399530026], {}),
    ("fake", [0.0011733893324340494, 0.6016126834507816], {}),
    ("mul", [0.9307150630961242, 0.2943025202085412], {}),
    ("div", [0.6545197868355805, 0.11276464241684814], {}),
    ("mul", [0.6573680105483782, 0.13653818666573825], {}),
    ("add", [0.7959397704608381, 0.41576468218269147], {}),
    ("mul", [0.16347738503061415, 0.04898545483226813], {}),
    ("fake", [0.7815677830141265, 0.013945854620984632], {}),
    ("fake", [0.8020187012792446, 0.25875661742111045], {}),
    ("fake", [0.9043915109893543, 0.6876522434184933], {}),
    ("add", [0.929922781635924, 0.9540258004221696], {}),
    ("fake", [0.961238888823737, 0.4030318469598062], {}),
    ("div", [0.7466755564012741, 0.799944178434153], {}),
    ("mul", [0.15308290555092918, 0.45297088397324015], {}),
    ("mul", [0.3505532310800745, 0.5551384076337974], {}),
]

In [44]:
client.check_registry = "never"
client.set_default_queue("cola_1")

fs = client.rpc_batch_async(tp)

In [45]:
# NBVAL_CHECK_OUTPUT

[f.safe_get() for f in fs]

[0.017299753708316164,
 None,
 0.17357941751386985,
 0.11098481677579876,
 0.6573534578300718,
 0.6040523459921044,
 None,
 None,
 None,
 0.5405443999702914,
 1.4029300451835836,
 0.27692063003323986,
 1.1001586089982087,
 None,
 1.1863498083998572,
 1.0892637063407844,
 None,
 0.2739117886652408,
 5.804299759281539,
 0.08975583613233945,
 1.2117044526435294,
 0.008008014060514455,
 None,
 None,
 None,
 1.8839485820580935,
 None,
 0.9334095759817276,
 0.06934209904859642,
 0.1946055624926752]

In [46]:
# NBVAL_CHECK_OUTPUT

client.responses

{}

In [47]:
client.check_registry = "cache"

In [48]:
# NBVAL_CHECK_OUTPUT

try:
    x = client.rpc_batch_sync(tp, timeout=5)
except Exception as e:
    print(e)

Method fake does not exist/is not available.


In [49]:
client.check_registry = "never"
client.set_default_queue("cola_1")

x = client.rpc_async("kk", [1, 0])

In [50]:
# NBVAL_CHECK_OUTPUT

try:
    x.get()
except Exception as e:
    print(e)

Error -32601 : The method does not exist/is not available.

 NoneType: None



In [51]:
y = client.rpc_async("add", [1, 0])

In [52]:
# NBVAL_CHECK_OUTPUT

y.get(5)

1

In [53]:
# NBVAL_CHECK_OUTPUT


def f(x, y):
    return x + y


# "never" use queue "default" don't work for rpc_async_fn or rpc_sync_fn
client.check_registry = "never"
y = client.rpc_async_fn(f, [1, 2.0])
try:
    print(y.get())
except Exception as e:
    print(e)

Error -32601 : The method does not exist/is not available.

 NoneType: None



In [54]:
client.check_registry = "cache"
y = client.rpc_async_fn(f, [1, 2.0])

In [55]:
# NBVAL_CHECK_OUTPUT

y.get()

3.0

In [56]:
# NBVAL_CHECK_OUTPUT

[k for k in client.responses.keys()]

[]

In [57]:
client.clean_used()

In [58]:
# NBVAL_CHECK_OUTPUT

[k for k in client.responses.keys()]

[]

In [59]:
# client.rpc_sync_fn(print, ["hola"])

In [60]:
# NBVAL_CHECK_OUTPUT

tp = [
    ("sleep", [15], {}),
    ("sleep", [15], {}),
    ("sleep", [15], {}),
    ("sleep", [15], {}),
    ("sleep", [15], {}),
    ("sleep", [15], {}),
    ("sleep", [15], {}),
    ("sleep", [15], {}),
    ("sleep", [15], {}),
    ("sleep", [15], {}),
]
tp

[('sleep', [15], {}),
 ('sleep', [15], {}),
 ('sleep', [15], {}),
 ('sleep', [15], {}),
 ('sleep', [15], {}),
 ('sleep', [15], {}),
 ('sleep', [15], {}),
 ('sleep', [15], {}),
 ('sleep', [15], {}),
 ('sleep', [15], {})]

In [61]:
# NBVAL_CHECK_OUTPUT

client.default_queue_ref

'requests_cola_1'

In [62]:
client.check_registry = "never"
fs = client.rpc_multi_async(tp, retry=True)

In [63]:
# NBVAL_CHECK_OUTPUT

gather(fs, 30, 5)

In [64]:
s = [f.status for f in fs]

In [65]:
# NBVAL_CHECK_OUTPUT

len(s), "OK" in s

(10, True)

In [66]:
[
    (time.time() if f.finished_time is None else f.finished_time) - f.creation_time
    for f in fs
]

[15.10536789894104,
 16.28783369064331,
 16.706504583358765,
 30.045740365982056,
 30.96531081199646,
 30.941241025924683,
 30.920538187026978,
 30.8974711894989,
 30.877833127975464,
 30.85350227355957]

In [67]:
[f.finished_time for f in fs]

[1774608647.6164353,
 1774608648.8267477,
 1774608649.270512,
 1774608662.6385326,
 None,
 None,
 None,
 None,
 None,
 None]

In [68]:
client.pending

{'fs_client_4:170': 1774608632.6158946,
 'fs_client_4:171': 1774608632.6399665,
 'fs_client_4:172': 1774608632.6606708,
 'fs_client_4:173': 1774608632.6837366,
 'fs_client_4:174': 1774608632.7033732,
 'fs_client_4:175': 1774608632.7277086}

In [69]:
fs[3].status

'OK'

In [70]:
fs[4].retry()

True

In [71]:
# [f.retry() for f in fs if not f.done()]
# no hace falta chequear si está pendiente.
[f.retry() for f in fs]

[False, False, False, False, True, True, True, True, True, True]

In [72]:
client.check_registry = "always"

In [73]:
# NBVAL_CHECK_OUTPUT

sorted(client.connector.all_queues_for_method("info"))

['requests_fs_server_4', 'requests_fs_server_5', 'requests_fs_server_6']

In [74]:
client.update_registry_cache()

In [75]:
# NBVAL_CHECK_OUTPUT

client.check_registry = "Never"
client._all_queue_refs_for_method("hola")

['requests_cola_1']

In [76]:
# NBVAL_CHECK_OUTPUT

client.check_registry = "always"
a = client.rpc_async("info")

In [77]:
# client.rpc_sync("div", [2, 1], timeout=10)

In [78]:
x = a.get()
x

('fs_server_5',
 {'requests_fs_server_5': {'info'},
  'requests_cola_1': {'add', 'dic', 'div', 'lista', 'mul', 'sleep', 'tupla'},
  'requests_cola_2': {'hola'},
  'requests_py_eval': {'eval_py_function'}})

In [79]:
# NBVAL_CHECK_OUTPUT

len(x), len(x[1])

(2, 4)

In [80]:
# NBVAL_CHECK_OUTPUT

x[1]["requests_cola_1"]

{'add', 'dic', 'div', 'lista', 'mul', 'sleep', 'tupla'}

In [81]:
# NBVAL_CHECK_OUTPUT

x[1]["requests_cola_2"]

{'hola'}

In [82]:
# NBVAL_CHECK_OUTPUT

x[1]["requests_py_eval"]

{'eval_py_function'}

In [83]:
# NBVAL_CHECK_OUTPUT

client.default_queue_ref

'requests_cola_1'

In [84]:
client.set_default_queue("cola_1")

In [85]:
# NBVAL_CHECK_OUTPUT

client.default_queue_ref

'requests_cola_1'

In [86]:
if LAUNCH_SERVER:
    client.rpc_sync("kill_all_processes", [], queue=MASTER_QUEUE, timeout=120)
    if platform.system() == "Windows":
        subprocess.run(
            ["taskkill", "/F", "/T", "/PID", str(server_process.pid)],
            capture_output=True,
        )

    else:
        server_process.terminate()
        server_process.wait()

In [87]:
# NBVAL_CHECK_OUTPUT

ps = client.rpc_async("list_processes", [], queue=MASTER_QUEUE)
ps.safe_get(3)

In [88]:
# NBVAL_CHECK_OUTPUT
try:
    y = client.rpc_async("add", [1, 0])
    y.safe_get(3)
except KeyError:
    print("Error")